# RLHF summarisation: SFT -> Reward -> PPO -> DPO
Each cell writes one file, then the last cell runs them in order.

In [ ]:
!pip install -q torch transformers datasets rouge-score matplotlib
import os; os.makedirs('starter',exist_ok=True); os.makedirs('results',exist_ok=True)

## `starter/config.py`

In [ ]:
%%writefile starter/config.py
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M"   # base model for every stage
SFT_DATA   = "CarperAI/openai_summarize_tldr"
PREF_DATA  = "CarperAI/openai_summarize_comparisons"

MAX_PROMPT_TOKENS  = 448      # post + "TL;DR:" longer than this is dropped
MAX_SUMMARY_TOKENS = 48       # summaries longer than this are dropped

SEED = 42
N_SFT_TRAIN, N_SFT_VAL   = 20_000, 500
N_PREF_TRAIN, N_PREF_VAL = 20_000, 1_000
N_EVAL_POSTS             = 300


## `starter/models.py`

In [ ]:
%%writefile starter/models.py
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from config import MODEL_NAME

def load_tokenizer(name=MODEL_NAME):
    tok = AutoTokenizer.from_pretrained(name)
    if tok.pad_token is None:                # SmolLM2 has no pad token
        tok.pad_token = tok.eos_token
    return tok

def load_causal_lm(name_or_path=MODEL_NAME, device=None):
    """Plain pretrained causal LM (no heads added). Use it for the SFT
    policy, the frozen reference model, and as the backbone of the
    reward model and of the PPO value head."""
    device = device or ("cuda" if torch.cuda.is_available() else "cpu")
    model = AutoModelForCausalLM.from_pretrained(name_or_path, dtype=torch.float32)
    return model.to(device)


## `starter/data.py`

In [ ]:
%%writefile starter/data.py
import random
from datasets import load_dataset
from config import *

TLDR = "TL;DR:"

def clean_post(raw_prompt):
    """Keep SUBREDDIT/TITLE/POST, drop anything from 'TL;DR:' on,
    and end with one fixed marker the model learns to continue."""
    post = raw_prompt.split(TLDR)[0].strip()
    return post + "\n" + TLDR

def clean_summary(raw):
    """Strip a leading 'TL;DR:', cut stray text after <|endoftext|>,
    and start with exactly one space (it follows 'TL;DR:')."""
    text = raw.split("<|endoftext|>")[0].strip()
    if text.startswith(TLDR):
        text = text[len(TLDR):].strip()
    return " " + text

def _sample_valid(rows, n, tok, make, seed):
    """Shuffle rows with a fixed seed and keep the first n that pass
    the length filter. Each kept item is a dict of token-id lists."""
    order = list(range(len(rows)))
    random.Random(seed).shuffle(order)
    kept = []
    for start in range(0, len(order), 2000):
        for i in order[start:start + 2000]:
            item = make(rows[i], tok)
            if item is not None:
                kept.append(item)
                if len(kept) == n:
                    return kept
    return kept

def _make_sft(row, tok):
    prompt, summary = clean_post(row["prompt"]), clean_summary(row["label"])
    p = tok(prompt, add_special_tokens=False)["input_ids"]
    s = tok(summary, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(summary.strip()) == 0 or len(p) > MAX_PROMPT_TOKENS or len(s) > MAX_SUMMARY_TOKENS:
        return None
    return {"prompt_ids": p, "summary_ids": s, "prompt": prompt, "summary": summary}

def _make_pref(row, tok):
    prompt = clean_post(row["prompt"])
    ch, rj = clean_summary(row["chosen"]), clean_summary(row["rejected"])
    if ch == rj or not ch.strip() or not rj.strip():
        return None
    p  = tok(prompt, add_special_tokens=False)["input_ids"]
    c  = tok(ch, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    r  = tok(rj, add_special_tokens=False)["input_ids"] + [tok.eos_token_id]
    if len(p) > MAX_PROMPT_TOKENS or max(len(c), len(r)) > MAX_SUMMARY_TOKENS:
        return None
    return {"prompt_ids": p, "chosen_ids": c, "rejected_ids": r,
            "prompt": prompt, "chosen": ch, "rejected": rj}

def load_sft_splits(tok):
    ds = load_dataset(SFT_DATA)
    train = _sample_valid(ds["train"], N_SFT_TRAIN, tok, _make_sft, SEED)
    val   = _sample_valid(ds["valid"], N_SFT_VAL,   tok, _make_sft, SEED)
    return train, val

def load_preference_splits(tok):
    ds = load_dataset(PREF_DATA)
    train = _sample_valid(ds["train"],  N_PREF_TRAIN, tok, _make_pref, SEED)
    val   = _sample_valid(ds["valid1"], N_PREF_VAL,   tok, _make_pref, SEED)
    return train, val

def load_eval_posts(tok):
    """Held-out posts (with their human reference summary) used to
    compare SFT, PPO and DPO at the end. Never train on these."""
    ds = load_dataset(SFT_DATA)
    return _sample_valid(ds["test"], N_EVAL_POSTS, tok, _make_sft, SEED)


## `starter/batching.py`

In [ ]:
%%writefile starter/batching.py
import torch

def _pad(seqs, pad_id, side):
    L = max(len(s) for s in seqs)
    ids  = torch.full((len(seqs), L), pad_id, dtype=torch.long)
    mask = torch.zeros((len(seqs), L), dtype=torch.long)
    for i, s in enumerate(seqs):
        t = torch.tensor(s, dtype=torch.long)
        if side == "right":
            ids[i, :len(s)], mask[i, :len(s)] = t, 1
        else:                                   # left padding
            ids[i, L - len(s):], mask[i, L - len(s):] = t, 1
    return ids, mask

def collate_sft(batch, pad_id):
    """Right-padded prompt+summary. prompt_lens[i] = number of prompt
    tokens in row i, so summary tokens are positions prompt_lens[i] .. end."""
    seqs = [b["prompt_ids"] + b["summary_ids"] for b in batch]
    ids, mask = _pad(seqs, pad_id, "right")
    prompt_lens = torch.tensor([len(b["prompt_ids"]) for b in batch])
    return {"input_ids": ids, "attention_mask": mask, "prompt_lens": prompt_lens}

def collate_pref(batch, pad_id):
    """Two right-padded batches (prompt+chosen, prompt+rejected) that
    share the same prompt_lens."""
    ch, ch_mask = _pad([b["prompt_ids"] + b["chosen_ids"]   for b in batch], pad_id, "right")
    rj, rj_mask = _pad([b["prompt_ids"] + b["rejected_ids"] for b in batch], pad_id, "right")
    prompt_lens = torch.tensor([len(b["prompt_ids"]) for b in batch])
    return {"chosen_ids": ch, "chosen_mask": ch_mask,
            "rejected_ids": rj, "rejected_mask": rj_mask,
            "prompt_lens": prompt_lens}

def collate_prompts(batch, pad_id):
    """LEFT-padded prompts only, for model.generate() in PPO rollouts
    and evaluation. Left padding keeps every prompt's last token at the
    same position, which is where generation starts."""
    ids, mask = _pad([b["prompt_ids"] for b in batch], pad_id, "left")
    return {"input_ids": ids, "attention_mask": mask}


## `common.py`

In [ ]:
%%writefile common.py
# common.py - small helpers used by every stage
import sys
sys.path.append("starter")          # so we can import the starter files

import os, json, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from functools import partial
from torch.utils.data import DataLoader

from config import *
from models import load_tokenizer, load_causal_lm
from data import load_sft_splits, load_preference_splits, load_eval_posts
from batching import collate_sft, collate_pref, collate_prompts

device = "cuda" if torch.cuda.is_available() else "cpu"
os.makedirs("results", exist_ok=True)
os.makedirs("checkpoints", exist_ok=True)


def token_logprobs(model, input_ids, attention_mask):
    """log pi(x_t | x_<t) for every position t >= 1.  Shape (B, L-1)."""
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits   # (B, L, V)
    logp = F.log_softmax(logits[:, :-1], dim=-1)     # position t-1 predicts token t
    targets = input_ids[:, 1:]                       # the tokens that were predicted
    return logp.gather(2, targets.unsqueeze(2)).squeeze(2)


def summary_mask(attention_mask, prompt_lens):
    """1 on summary tokens, 0 on prompt and padding.  Shape (B, L-1),
    the same shape as token_logprobs (token 0 is dropped)."""
    L = attention_mask.shape[1]
    positions = torch.arange(L, device=attention_mask.device)        # 0, 1, ..., L-1
    after_prompt = positions[None, :] >= prompt_lens[:, None]        # not a prompt token
    mask = after_prompt & (attention_mask == 1)                      # and not padding
    return mask[:, 1:].float()


def masked_mean(x, mask):
    """Mean of x only where mask is 1."""
    return (x * mask).sum() / mask.sum()


@torch.no_grad()
def generate(model, prompt_ids, prompt_mask, sample=False):
    """Returns only the new tokens, shape (B, T) with T <= 48."""
    if sample:      # PPO rollout: pure sampling
        out = model.generate(input_ids=prompt_ids, attention_mask=prompt_mask,
                             max_new_tokens=MAX_SUMMARY_TOKENS, do_sample=True,
                             temperature=1.0, top_k=0, top_p=1.0,
                             pad_token_id=model.config.eos_token_id)
    else:           # greedy
        out = model.generate(input_ids=prompt_ids, attention_mask=prompt_mask,
                             max_new_tokens=MAX_SUMMARY_TOKENS, do_sample=False,
                             pad_token_id=model.config.eos_token_id)
    return out[:, prompt_ids.shape[1]:]


def generated_mask(new_tokens, eos_id):
    """1 up to and including the first EOS, 0 after it.  Shape (B, T)."""
    is_eos = (new_tokens == eos_id).long()
    eos_before = is_eos.cumsum(1) - is_eos        # how many EOS came before this position
    return (eos_before == 0).float()


def save_plot(x, curves, xlabel, path):
    """curves = {"name": list of values}.  One small plot per curve."""
    fig, axes = plt.subplots(1, len(curves), figsize=(5 * len(curves), 3.5))
    if len(curves) == 1:
        axes = [axes]
    for ax, name in zip(axes, curves):
        ax.plot(x, curves[name])
        ax.set_xlabel(xlabel)
        ax.set_title(name)
    plt.tight_layout()
    plt.savefig(path)
    plt.close()


# ---------------- Task 1: data report + mask check ----------------
if __name__ == "__main__":
    tok = load_tokenizer()
    sft_train, sft_val = load_sft_splits(tok)
    pref_train, pref_val = load_preference_splits(tok)
    eval_posts = load_eval_posts(tok)

    p_lens = [len(ex["prompt_ids"]) for ex in sft_train]
    s_lens = [len(ex["summary_ids"]) for ex in sft_train]

    table = "| | |\n|---|---|\n"
    table += f"| SFT train / validation pairs | {len(sft_train)} / {len(sft_val)} |\n"
    table += f"| Preference train / validation pairs | {len(pref_train)} / {len(pref_val)} |\n"
    table += f"| Evaluation posts | {len(eval_posts)} |\n"
    table += f"| Mean / max prompt length, SFT train (tokens) | {sum(p_lens) / len(p_lens):.1f} / {max(p_lens)} |\n"
    table += f"| Mean / max summary length, SFT train (tokens) | {sum(s_lens) / len(s_lens):.1f} / {max(s_lens)} |\n"
    print(table)
    open("results/table1_data.md", "w").write(table)

    # check: decoding the tokens where summary_mask is 1 must give the summary + EOS
    model = load_causal_lm()
    b = collate_sft(sft_train[:4], tok.pad_token_id)
    ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
    with torch.no_grad():
        logp = token_logprobs(model, ids, mask)
    smask = summary_mask(mask, b["prompt_lens"].to(device))
    print("token_logprobs shape:", tuple(logp.shape), " input shape:", tuple(ids.shape))
    for i in range(4):
        tokens = ids[i, 1:][smask[i] == 1]          # ids[:, 1:] lines up with the mask
        print("MASKED :", repr(tok.decode(tokens)))
        print("SUMMARY:", repr(sft_train[i]["summary"] + tok.eos_token))


## `train_sft.py`

In [ ]:
%%writefile train_sft.py
# train_sft.py - Task 2: supervised fine-tuning
from common import *

LR, BATCH, LOG_EVERY = 5e-5, 16, 250

torch.manual_seed(SEED)
tok = load_tokenizer()
pad = tok.pad_token_id
sft_train, sft_val = load_sft_splits(tok)
eval_posts = load_eval_posts(tok)

train_loader = DataLoader(sft_train, batch_size=BATCH, shuffle=True, collate_fn=partial(collate_sft, pad_id=pad))
val_loader = DataLoader(sft_val, batch_size=BATCH, collate_fn=partial(collate_sft, pad_id=pad))

model = load_causal_lm()
opt = torch.optim.AdamW(model.parameters(), lr=LR)


def summary_logprobs(b):
    """Log-probs and the summary mask for one batch."""
    ids, mask = b["input_ids"].to(device), b["attention_mask"].to(device)
    logp = token_logprobs(model, ids, mask)
    smask = summary_mask(mask, b["prompt_lens"].to(device))
    return logp, smask


@torch.no_grad()
def validation_loss():
    total, count = 0.0, 0.0
    for b in val_loader:
        logp, smask = summary_logprobs(b)
        total += -(logp * smask).sum().item()
        count += smask.sum().item()
    return total / count


def show_samples(title):
    """Greedy summaries for the first 5 evaluation posts."""
    b = collate_prompts(eval_posts[:5], pad)
    new = generate(model, b["input_ids"].to(device), b["attention_mask"].to(device))
    text = f"## {title}\n\n"
    for i in range(5):
        text += f"{i + 1}. {tok.decode(new[i], skip_special_tokens=True).strip()}\n"
    print(text)
    return text + "\n"


samples = show_samples("Before SFT")

log = []
start = time.time()
for step, b in enumerate(train_loader, 1):
    logp, smask = summary_logprobs(b)
    loss = -masked_mean(logp, smask)          # mean NLL of the summary tokens only

    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    if step % LOG_EVERY == 0 or step == len(train_loader):
        val = validation_loss()
        log.append({"step": step, "train_loss": loss.item(), "val_loss": val})
        print(log[-1])
minutes = (time.time() - start) / 60

model.save_pretrained("checkpoints/sft")
samples += show_samples("After SFT")
open("results/sft_samples.md", "w", encoding="utf-8").write(samples)
json.dump(log, open("results/sft_log.json", "w"))

steps = [r["step"] for r in log]
save_plot(steps, {"SFT train loss": [r["train_loss"] for r in log],
                  "SFT validation loss": [r["val_loss"] for r in log]}, "step", "results/fig1_sft.png")

row = f"| SFT | {log[-1]['train_loss']:.4f} | val. loss: {log[-1]['val_loss']:.4f} | {minutes:.1f} min |\n"
open("results/table2_training.md", "a").write(row)
print(row)


## `train_reward.py`

In [ ]:
%%writefile train_reward.py
# train_reward.py - Task 3: reward model
from common import *

LR, BATCH, LOG_EVERY = 1e-5, 8, 250


class RewardModel(nn.Module):
    """Transformer body (no LM head) + one linear layer that gives a score."""

    def __init__(self, path="checkpoints/sft"):
        super().__init__()
        self.body = load_causal_lm(path).model                    # drop the LM head
        self.head = nn.Linear(self.body.config.hidden_size, 1).to(device)

    def forward(self, ids, mask):
        hidden = self.body(input_ids=ids, attention_mask=mask).last_hidden_state   # (B, L, H)
        positions = torch.arange(ids.shape[1], device=ids.device)
        last = (mask * positions).argmax(1)             # index of the last real token
        rows = torch.arange(ids.shape[0], device=ids.device)
        return self.head(hidden[rows, last]).squeeze(1)  # (B,)


def load_reward_model():
    """The trained reward model, frozen."""
    rm = RewardModel()
    rm.load_state_dict(torch.load("checkpoints/reward/reward.pt", map_location=device))
    for p in rm.parameters():
        p.requires_grad = False
    return rm


if __name__ == "__main__":
    torch.manual_seed(SEED)
    tok = load_tokenizer()
    pad = tok.pad_token_id
    pref_train, pref_val = load_preference_splits(tok)
    train_loader = DataLoader(pref_train, batch_size=BATCH, shuffle=True, collate_fn=partial(collate_pref, pad_id=pad))
    val_loader = DataLoader(pref_val, batch_size=BATCH, collate_fn=partial(collate_pref, pad_id=pad))

    rm = RewardModel()
    opt = torch.optim.AdamW(rm.parameters(), lr=LR)

    def scores(b):
        r_w = rm(b["chosen_ids"].to(device), b["chosen_mask"].to(device))        # winner
        r_l = rm(b["rejected_ids"].to(device), b["rejected_mask"].to(device))    # loser
        return r_w, r_l

    @torch.no_grad()
    def validation():
        """Pairwise accuracy, and the list True/False (correct or not) for every pair."""
        correct = []
        for b in val_loader:
            r_w, r_l = scores(b)
            correct += (r_w > r_l).tolist()
        return sum(correct) / len(correct), correct

    log = []
    start = time.time()
    for step, b in enumerate(train_loader, 1):
        r_w, r_l = scores(b)
        loss = -F.logsigmoid(r_w - r_l).mean()        # Bradley-Terry loss

        opt.zero_grad()
        loss.backward()
        opt.step()

        if step % LOG_EVERY == 0 or step == len(train_loader):
            acc, correct = validation()
            log.append({"step": step, "train_loss": loss.item(), "val_acc": acc})
            print(log[-1])
    minutes = (time.time() - start) / 60

    os.makedirs("checkpoints/reward", exist_ok=True)
    torch.save(rm.state_dict(), "checkpoints/reward/reward.pt")
    json.dump(log, open("results/reward_log.json", "w"))

    steps = [r["step"] for r in log]
    save_plot(steps, {"Reward model train loss": [r["train_loss"] for r in log],
                      "Reward model validation accuracy": [r["val_acc"] for r in log]}, "step", "results/fig2_reward.png")

    # three validation pairs the reward model gets wrong (for Task 7.1)
    wrong = [pref_val[i] for i in range(len(pref_val)) if not correct[i]][:3]
    text = "# Three pairs the reward model gets wrong\n\n"
    for ex in wrong:
        text += f"**Post (end):** ...{ex['prompt'][-400:]}\n\n**Human chose:**{ex['chosen']}\n\n**Human rejected:**{ex['rejected']}\n\n---\n\n"
    open("results/reward_mistakes.md", "w", encoding="utf-8").write(text)

    row = f"| Reward model | {log[-1]['train_loss']:.4f} | pairwise acc.: {log[-1]['val_acc']:.4f} | {minutes:.1f} min |\n"
    open("results/table2_training.md", "a").write(row)
    print(row)


## `train_ppo.py`

In [ ]:
%%writefile train_ppo.py
# train_ppo.py - Task 4: PPO
from common import *
from train_reward import load_reward_model

CLIP_EPS, BETA_KL, C_V, C_H, GAMMA = 0.2, 0.05, 0.1, 0.01, 1.0
K_EPOCHS, LR, BATCH, ITERATIONS = 4, 1e-5, 16, 200

torch.manual_seed(SEED)
tok = load_tokenizer()
pad = tok.pad_token_id
sft_train, sft_val = load_sft_splits(tok)

# ---- three models in memory ----
policy = load_causal_lm("checkpoints/sft")                             # trained
value_head = nn.Linear(policy.config.hidden_size, 1).to(device)        # trained
ref = load_causal_lm("checkpoints/sft")                                # frozen
reward_model = load_reward_model()                                     # frozen
for p in ref.parameters():
    p.requires_grad = False
print("ref frozen   :", all(not p.requires_grad for p in ref.parameters()))
print("reward frozen:", all(not p.requires_grad for p in reward_model.parameters()))

params = list(policy.parameters()) + list(value_head.parameters())
opt = torch.optim.AdamW(params, lr=LR)


def policy_forward(ids, mask, P):
    """For every generated token a_t: log pi(a_t|s_t), V(s_t), entropy.  Each (B, T).
    P = prompt length. Token at position P+k is predicted by position P+k-1,
    so we read logits and hidden states from positions P-1 ... L-2."""
    out = policy(input_ids=ids, attention_mask=mask, output_hidden_states=True)
    logits = out.logits[:, P - 1:-1]                       # (B, T, V)
    hidden = out.hidden_states[-1][:, P - 1:-1]            # (B, T, H) last hidden layer
    logp_all = F.log_softmax(logits, dim=-1)
    actions = ids[:, P:]                                   # the generated tokens
    logp = logp_all.gather(2, actions.unsqueeze(2)).squeeze(2)
    values = value_head(hidden).squeeze(2)
    entropy = -(logp_all.exp() * logp_all).sum(-1)
    return logp, values, entropy


log = []
start = time.time()
for it in range(ITERATIONS):
    # ---------- 1. rollout: sample one summary for 16 prompts ----------
    b = collate_prompts(sft_train[it * BATCH:(it + 1) * BATCH], pad)
    p_ids, p_mask = b["input_ids"].to(device), b["attention_mask"].to(device)
    P = p_ids.shape[1]

    with torch.no_grad():
        new = generate(policy, p_ids, p_mask, sample=True)          # (B, T)
        g_mask = generated_mask(new, tok.eos_token_id)              # 1 until first EOS
        ids = torch.cat([p_ids, new], dim=1)
        mask = torch.cat([p_mask, g_mask.long()], dim=1)
        T = new.shape[1]

        # ---------- 2. store old values (no gradients) ----------
        old_logp, old_values, _ = policy_forward(ids, mask, P)
        ref_logp = token_logprobs(ref, ids, mask)[:, P - 1:]
        score = reward_model(ids, mask)                             # (B,)

        # ---------- 3. per-token reward ----------
        kl = (old_logp - ref_logp) * g_mask
        rewards = -BETA_KL * kl
        last = g_mask.sum(1).long() - 1                             # last token of each summary
        rewards[torch.arange(len(last)), last] += score

        # ---------- 4. returns and advantages ----------
        returns = torch.zeros_like(rewards)
        running = torch.zeros(len(last), device=device)
        for t in reversed(range(T)):
            running = rewards[:, t] + GAMMA * running
            returns[:, t] = running
        adv = returns - old_values
        real = g_mask.bool()
        adv = (adv - adv[real].mean()) / (adv[real].std() + 1e-8)

    # ---------- 5. K epochs of PPO updates on the same rollout ----------
    clip_fracs = []
    for epoch in range(K_EPOCHS):
        logp, values, entropy = policy_forward(ids, mask, P)
        ratio = torch.exp(logp - old_logp)
        clipped = torch.clamp(ratio, 1 - CLIP_EPS, 1 + CLIP_EPS)

        l_ppo = masked_mean(torch.min(ratio * adv, clipped * adv), g_mask)
        l_value = masked_mean((values - returns) ** 2, g_mask)
        l_entropy = masked_mean(entropy, g_mask)
        loss = -l_ppo + C_V * l_value - C_H * l_entropy

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step()

        outside = ((ratio < 1 - CLIP_EPS) | (ratio > 1 + CLIP_EPS)).float()
        clip_fracs.append(masked_mean(outside, g_mask).item())

    # ---------- 6. logging ----------
    log.append({"iter": it + 1,
                "score": score.mean().item(),
                "kl": kl.sum(1).mean().item(),
                "length": g_mask.sum(1).mean().item(),
                "clip_frac": sum(clip_fracs) / K_EPOCHS,
                "clip_frac_epoch1": clip_fracs[0],
                "l_ppo": l_ppo.item(), "l_value": l_value.item(), "l_entropy": l_entropy.item()})
    print(log[-1])
minutes = (time.time() - start) / 60

# ---------- 7. save the policy only (no value head) ----------
policy.save_pretrained("checkpoints/ppo")
json.dump(log, open("results/ppo_log.json", "w"))

iters = [r["iter"] for r in log]
save_plot(iters, {"PPO mean reward score": [r["score"] for r in log],
                  "PPO mean KL to reference": [r["kl"] for r in log],
                  "PPO mean length": [r["length"] for r in log],
                  "PPO clip fraction": [r["clip_frac"] for r in log]}, "iteration", "results/fig3_ppo.png")

final_score = sum(r["score"] for r in log[-10:]) / len(log[-10:])     # mean of the last 10 iterations
row = f"| PPO | {loss.item():.4f} | final mean r: {final_score:.4f} | {minutes:.1f} min |\n"
open("results/table2_training.md", "a").write(row)
print(row)


## `train_dpo.py`

In [ ]:
%%writefile train_dpo.py
# train_dpo.py - Task 5: DPO
from common import *

BETA, LR, BATCH, LOG_EVERY = 0.1, 2e-6, 8, 250

torch.manual_seed(SEED)
tok = load_tokenizer()
pad = tok.pad_token_id
pref_train, pref_val = load_preference_splits(tok)
train_loader = DataLoader(pref_train, batch_size=BATCH, shuffle=True, collate_fn=partial(collate_pref, pad_id=pad))
val_loader = DataLoader(pref_val, batch_size=BATCH, collate_fn=partial(collate_pref, pad_id=pad))

policy = load_causal_lm("checkpoints/sft")       # trained
ref = load_causal_lm("checkpoints/sft")          # frozen
for p in ref.parameters():
    p.requires_grad = False
print("ref frozen:", all(not p.requires_grad for p in ref.parameters()))

opt = torch.optim.AdamW(policy.parameters(), lr=LR)


def seq_logprob(model, ids, mask, prompt_lens):
    """log pi(y | x) = sum of the log-probs of the summary tokens.  Shape (B,)."""
    logp = token_logprobs(model, ids, mask)
    return (logp * summary_mask(mask, prompt_lens)).sum(1)


def dpo_margin(b):
    """The expression inside sigma(.) in the DPO loss.  Shape (B,)."""
    w_ids, w_mask = b["chosen_ids"].to(device), b["chosen_mask"].to(device)
    l_ids, l_mask = b["rejected_ids"].to(device), b["rejected_mask"].to(device)
    plens = b["prompt_lens"].to(device)

    pi_w = seq_logprob(policy, w_ids, w_mask, plens)
    pi_l = seq_logprob(policy, l_ids, l_mask, plens)
    with torch.no_grad():
        ref_w = seq_logprob(ref, w_ids, w_mask, plens)
        ref_l = seq_logprob(ref, l_ids, l_mask, plens)
    return BETA * (pi_w - ref_w) - BETA * (pi_l - ref_l)


@torch.no_grad()
def validation_accuracy():
    """Implicit reward accuracy: share of pairs where the margin is positive."""
    correct = []
    for b in val_loader:
        correct += (dpo_margin(b) > 0).tolist()
    return sum(correct) / len(correct)


log = []
start = time.time()
for step, b in enumerate(train_loader, 1):
    loss = -F.logsigmoid(dpo_margin(b)).mean()
    if step == 1:
        print("first step loss (should be 0.6931):", loss.item())

    opt.zero_grad()
    loss.backward()
    opt.step()

    if step % LOG_EVERY == 0 or step == len(train_loader):
        acc = validation_accuracy()
        log.append({"step": step, "train_loss": loss.item(), "val_acc": acc})
        print(log[-1])
minutes = (time.time() - start) / 60

policy.save_pretrained("checkpoints/dpo")
json.dump(log, open("results/dpo_log.json", "w"))

steps = [r["step"] for r in log]
save_plot(steps, {"DPO train loss": [r["train_loss"] for r in log],
                  "DPO implicit reward accuracy": [r["val_acc"] for r in log]}, "step", "results/fig4_dpo.png")

row = f"| DPO | {log[-1]['train_loss']:.4f} | implicit reward acc.: {log[-1]['val_acc']:.4f} | {minutes:.1f} min |\n"
open("results/table2_training.md", "a").write(row)
print(row)


## `evaluate.py`

In [ ]:
%%writefile evaluate.py
# evaluate.py - Task 6: compare SFT, PPO and DPO on the 300 evaluation posts
from common import *
from train_reward import load_reward_model
from rouge_score import rouge_scorer

BATCH = 16
NAMES = ["sft", "ppo", "dpo"]

tok = load_tokenizer()
pad = tok.pad_token_id
eval_posts = load_eval_posts(tok)
n = len(eval_posts)

ref = load_causal_lm("checkpoints/sft")          # KL is measured against SFT
reward_model = load_reward_model()
rouge = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

# res["ppo"]["score"] is a list with one number per post, and so on
res = {name: {"summary": [], "score": [], "kl": [], "length": [], "rouge": []} for name in NAMES}

for name in NAMES:
    model = load_causal_lm("checkpoints/" + name)
    for i in range(0, n, BATCH):
        posts = eval_posts[i:i + BATCH]
        b = collate_prompts(posts, pad)
        p_ids, p_mask = b["input_ids"].to(device), b["attention_mask"].to(device)
        P = p_ids.shape[1]

        with torch.no_grad():
            new = generate(model, p_ids, p_mask)                      # greedy
            g_mask = generated_mask(new, tok.eos_token_id)
            ids = torch.cat([p_ids, new], dim=1)
            mask = torch.cat([p_mask, g_mask.long()], dim=1)

            score = reward_model(ids, mask)
            logp = token_logprobs(model, ids, mask)[:, P - 1:]        # this policy
            ref_logp = token_logprobs(ref, ids, mask)[:, P - 1:]      # SFT
            kl = ((logp - ref_logp) * g_mask).sum(1)

        for j in range(len(posts)):
            text = tok.decode(new[j], skip_special_tokens=True).strip()
            res[name]["summary"].append(text)
            res[name]["score"].append(score[j].item())
            res[name]["kl"].append(kl[j].item())
            res[name]["length"].append(g_mask[j].sum().item())
            res[name]["rouge"].append(rouge.score(posts[j]["summary"], text)["rougeL"].fmeasure)
    print("done:", name)


def mean(xs):
    return sum(xs) / len(xs)


# ---- Table 3 ----
table = "| Policy | Mean r | Win rate vs. SFT | KL to SFT | Length | ROUGE-L |\n|---|---|---|---|---|---|\n"
for name in NAMES:
    r = res[name]
    if name == "sft":
        win = "-"
    else:
        wins = [r["score"][i] > res["sft"]["score"][i] for i in range(n)]
        win = f"{mean(wins):.3f}"
    table += f"| {name.upper()} | {mean(r['score']):.4f} | {win} | {mean(r['kl']):.4f} | {mean(r['length']):.1f} | {mean(r['rouge']):.4f} |\n"
print(table)
open("results/table3_comparison.md", "w").write(table)

# ---- all summaries, one JSON object per line ----
with open("results/eval_summaries.jsonl", "w", encoding="utf-8") as f:
    for i in range(n):
        row = {"id": i, "prompt": eval_posts[i]["prompt"], "reference": eval_posts[i]["summary"].strip()}
        for name in NAMES:
            row[name] = res[name]["summary"][i]
            row[name + "_score"] = res[name]["score"][i]
        f.write(json.dumps(row) + "\n")

# ---- 10 qualitative samples (add your own comments in the file afterwards) ----
text = "# Qualitative samples\n\n"
for i in range(min(10, len(eval_posts))):
    text += f"## Post {i}\n\n...{eval_posts[i]['prompt'][-500:]}\n\n"
    text += f"- **Human:** {eval_posts[i]['summary'].strip()}\n"
    for name in NAMES:
        text += f"- **{name.upper()}** (r = {res[name]['score'][i]:.2f}): {res[name]['summary'][i]}\n"
    text += "- **Comment:** \n\n"
open("results/samples.md", "w", encoding="utf-8").write(text)


In [ ]:
for s in ['common.py','train_sft.py','train_reward.py','train_ppo.py','train_dpo.py','evaluate.py']:
    !python {s}